# Verify missingness of imaging data

In [ ]:
import pandas as pd
from preprocessing.geneva_stroke_unit_preprocessing.utils import create_registry_case_identification_column

In [ ]:
features_path = '/Users/jk1/temp/opsum_end/preprocessing/with_imaging/gsu_Extraction_20220815_prepro_30012026_154047/preprocessed_features_30012026_154047.csv'
registry_path = '/Users/jk1/stroke_datasets/stroke_registry_post_hoc_modified.xlsx'

In [ ]:
features_df = pd.read_csv(features_path)
registry_df = pd.read_excel(registry_path)

In [ ]:
registry_df['case_admission_id'] = create_registry_case_identification_column(registry_df)


In [ ]:
features_df.head()

In [ ]:
features_df.sample_label.unique()

In [ ]:
imaging_labels = ['cbf_lt_20',
       'cbf_lt_30', 'cbf_lt_34', 'cbf_lt_38', 'cbv_lt_34', 'cbv_lt_38',
       'cbv_lt_42', 'tmax_gt_10',
       'tmax_gt_4', 'tmax_gt_6', 'tmax_gt_8']

In [ ]:
imaging_df = features_df[features_df.sample_label.isin(imaging_labels)]
hypoperf_df = features_df[features_df.sample_label == 'hypoperfusion_with_mismatch']

In [ ]:
cids_with_imaging_according_to_hypoperfusion = hypoperf_df[hypoperf_df.source == 'stroke_registry'].case_admission_id.unique()

In [ ]:
cids_with_imaging = imaging_df[imaging_df.source == 'EHR'].case_admission_id.unique()
cids_without_imaging = set(features_df.case_admission_id.unique()) - set(cids_with_imaging)

print(f'Number of cases with imaging: {len(cids_with_imaging)}')
print(f'Number of cases without imaging: {len(cids_without_imaging)}')

In [ ]:
cids_with_missing_imaging_according_to_hypoperfusion = set(cids_with_imaging_according_to_hypoperfusion) - set(cids_with_imaging)
print(f'Number of cases with missing imaging according to hypoperfusion: {len(cids_with_missing_imaging_according_to_hypoperfusion)}')

In [ ]:
# create a dataframe with columsn case_admission_id and imaging_missing
missing_imaging_df = pd.DataFrame({'case_admission_id': list(features_df.case_admission_id.unique())})
missing_imaging_df['imaging_missing'] = missing_imaging_df.case_admission_id.apply(lambda x: 1 if x in cids_without_imaging else 0)
missing_imaging_df['imaging_missing_according_to_registry'] = missing_imaging_df.case_admission_id.apply(lambda x: 1 if x in cids_with_missing_imaging_according_to_hypoperfusion else 0)

In [ ]:
registry_df['year'] = registry_df['Entry date'].apply(lambda x: str(x)[0:4] if x else None).astype('Int64')

In [ ]:
missing_imaging_df = missing_imaging_df.merge(registry_df[['case_admission_id', 'year']], on='case_admission_id', how='left')

In [ ]:
# make a histogram of the years of the cases with missing imaging (overall and according to hypoperfusion
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.histplot(missing_imaging_df[missing_imaging_df.imaging_missing == 1]['year'], bins=range(2017, 2024), color='blue', label='Missing Imaging (EHR)', kde=False)
sns.histplot(missing_imaging_df[missing_imaging_df.imaging_missing_according_to_registry == 1]['year'], bins=range(2017, 2024), color='red', label='Missing Imaging (Hypoperfusion)', kde=False)
plt.xlabel('Year')
plt.ylabel('Count')
plt.title('Distribution of Cases with Missing Imaging by Year')

# add legend
plt.legend()
plt.show()

In [ ]:
missing_imaging_df[missing_imaging_df.imaging_missing == 1].year.value_counts().sort_index()

In [ ]:
registry_df[registry_df.case_admission_id.isin(cids_with_missing_imaging_according_to_hypoperfusion)]

In [ ]:
extraction_target_df = registry_df[registry_df.case_admission_id.isin(cids_with_missing_imaging_according_to_hypoperfusion)]
extraction_target_df['patient_id'] = extraction_target_df['Case ID'].apply(lambda x: x[8:-4]).astype(str)
# columns Case ID, case_admissions_id, patient_id
extraction_target_df = extraction_target_df[['Case ID', 'case_admission_id', 'patient_id']]

In [ ]:
# extraction_target_df.to_csv('/Users/jk1/Downloads/refined_extraction_target_20022026.csv', index=False)